# Notebook 03: Model Training — Statistical & ML Forecasting

## CRISP-DM Phase: Modeling

**Project**: OptiWMS — AI-Driven Warehouse Management System  
**Reference**: Petropoulos et al. (2022), Sections 2.3 (Exponential Smoothing), 2.5 (ARIMA), 2.7 (ML), 2.8 (Intermittent)

---

### Model Strategy (Enterprise-Grade)

| Model Type | Method | Use Case | Quantile Output |
|------------|--------|----------|----------------|
| Statistical | ETS (Holt-Winters) | Smooth seasonal demand | p10, p50, p90 via residual std |
| Statistical | ARIMA/SARIMAX | Trending + seasonal | p10, p50, p90 from confidence intervals |
| Statistical | Croston (SBA) | Intermittent demand (RM) | p50 + bootstrap intervals |
| ML | LightGBM | Large-scale, fast, handles categoricals | Quantile regression |
| ML | CatBoost | Excellent with categoricals | Quantile regression |
| ML | XGBoost | Robust baseline | Quantile regression |
| Baseline | Seasonal Naive | Repeat same month last year | - |

### Why ML for Warehousing?

> *"ML methods are particularly suited when cross-learning from a large number of related time series is beneficial."*  
> — Petropoulos et al. (2022), Section 2.7.4  
> *"The M5 competition confirmed that gradient-boosted trees outperform traditional methods on complex, hierarchical data."*

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import time

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAS_SM = True
except ImportError:
    HAS_SM = False

try:
    import mlflow
    import mlflow.lightgbm  # noqa: F401
    import mlflow.catboost  # noqa: F401
    import mlflow.xgboost  # noqa: F401
    HAS_MLFLOW = True
except ImportError:
    HAS_MLFLOW = False
    print('MLflow not installed — results will be logged locally only')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
SEED = 42
np.random.seed(SEED)

ROOT = Path('..').resolve()
V6_DIR = Path('.').resolve()
import sys
if str(V6_DIR) not in sys.path:
    sys.path.insert(0, str(V6_DIR))
from forecast_utils import (
    tune_lightgbm_optuna, auto_arima_forecast, sarimax_fallback_forecast,
    rolling_origin_splits, sku_sample_weights, wape as fu_wape,
    evaluate_forecast_suite, cap_forecast,
)
DATA_DIR = ROOT.parent / 'Forecast model train data optiwms'
GEN_DIR  = ROOT / 'outputs' / 'generated'
ENG_DIR  = ROOT / 'outputs' / 'engineered'

print('Libraries loaded successfully')

# Memory guards for local training (avoid machine crash)
MAX_TRAIN_ROWS = 50_000   # monthly panel ~3.7k; headroom if data grows
LGB_N_ESTIMATORS = 300    # was 500 — lower RAM peak
CAT_ITERATIONS = 300
XGB_N_ESTIMATORS = 300
LGB_THREADS = 2             # limit parallel threads to reduce RAM spikes



In [ ]:
# Load engineered features from NB02 (must be monthly aggregated — ~3-4k rows, not 222k)
import gc

if (ENG_DIR / 'fg_features_engineered.csv').exists():
    fg = pd.read_csv(ENG_DIR / 'fg_features_engineered.csv')
    fg['month'] = pd.to_datetime(fg['month'])
    print(f'Loaded engineered features: {fg.shape}')
    if len(fg) > 20_000:
        print('WARNING: Old bloated engineered file — re-aggregating to monthly SKU panel...')
        if 'demand_units' not in fg.columns and 'demand_units_clean' in fg.columns:
            fg['demand_units'] = fg['demand_units_clean']
        fg['month'] = fg['month'].dt.to_period('M').dt.to_timestamp()
        num_cols = fg.select_dtypes(include=[np.number]).columns.tolist()
        agg = {c: 'mean' for c in num_cols if c != 'demand_units'}
        if 'demand_units' in fg.columns:
            agg['demand_units'] = 'mean'  # 60 MC scenarios/SKU-month — mean = expected demand (sum would 60x inflate)
        for c in [c for c in fg.columns if c not in agg and c not in ('fg_code', 'month')]:
            agg[c] = 'first'
        fg = fg.groupby(['fg_code', 'month'], as_index=False).agg(agg)
        print(f'Reduced to monthly panel: {fg.shape}')
else:
    print('Run NB02 first. Loading raw data with monthly aggregation fallback...')
    fg = pd.read_csv(DATA_DIR / 'hemas_scenario_c_dataset_cleaned.csv')
    fg['month'] = pd.to_datetime(fg['month']).dt.to_period('M').dt.to_timestamp()
    if 'demand_units_clean' in fg.columns:
        fg['demand_units'] = fg['demand_units_clean']
    num_cols = fg.select_dtypes(include=[np.number]).columns.tolist()
    agg = {c: 'mean' for c in num_cols if c != 'demand_units'}
    agg['demand_units'] = 'mean'  # 60 MC scenarios/SKU-month — mean = expected demand (sum would 60x inflate)
    fg = fg.groupby(['fg_code', 'month'], as_index=False).agg(agg)

if len(fg) > 20_000:
    raise MemoryError(
        f'Dataset has {len(fg):,} rows — expected ~3,700 monthly rows. Re-run NB02 to aggregate to SKU-month.')

for col in fg.select_dtypes(include=['float64']).columns:
    fg[col] = fg[col].astype('float32')
gc.collect()

print(f'SKUs: {fg["fg_code"].nunique()}, Rows: {len(fg):,}, RAM-safe for training')


## 1. MLflow Experiment Setup

> *"Experiment tracking is essential for reproducibility."* — ML Pipelines Module, Week 3  
> *"MLflow provides model versioning, comparison, and deployment."* — MLflow Module, Week 6

In [ ]:
# MLflow setup
import urllib.request

EXPERIMENT_NAME = 'optiwms-demand-forecast-v6'
MLFLOW_SERVER_VERSION = '2.16.0'  # match infra/docker-compose.mlops.yml image

MLFLOW_TRACKING_URI = None
for uri in ('http://localhost:5001', 'http://localhost:5000'):
    try:
        urllib.request.urlopen(f'{uri}/health', timeout=3)
        MLFLOW_TRACKING_URI = uri
        break
    except Exception:
        continue

def mlflow_log_model_safe(model, artifact_path, flavor):
    """Log model artifact; metrics/params still saved if this fails."""
    try:
        if flavor == 'lightgbm':
            mlflow.lightgbm.log_model(model, artifact_path)
        elif flavor == 'catboost':
            mlflow.catboost.log_model(model, artifact_path)
        elif flavor == 'xgboost':
            mlflow.xgboost.log_model(model, artifact_path)
        else:
            mlflow.sklearn.log_model(model, artifact_path)
        print(f'  Model artifact logged ({flavor})')
    except Exception as e:
        print(f'  Metrics/params saved; model artifact skipped: {e}')

if HAS_MLFLOW:
    try:
        client_ver = tuple(int(x) for x in mlflow.__version__.split('.')[:2])
        server_ver = tuple(int(x) for x in MLFLOW_SERVER_VERSION.split('.')[:2])
        if client_ver != server_ver:
            print(f'Warning: MLflow client {mlflow.__version__} != server {MLFLOW_SERVER_VERSION}')
            print('  Pin client when disk space allows: pip install "mlflow==2.16.0"')
            print('  Metrics/params will still log; model artifacts may be skipped.')
        if MLFLOW_TRACKING_URI is None:
            raise ConnectionError('No MLflow server on localhost:5001 or :5000')
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        mlflow.set_experiment(EXPERIMENT_NAME)
        print(f'MLflow connected: {MLFLOW_TRACKING_URI}')
        print(f'Experiment: {EXPERIMENT_NAME}')
        print(f'UI: {MLFLOW_TRACKING_URI}')
    except Exception as e:
        print(f'MLflow server not reachable ({e}). Logging locally.')
        HAS_MLFLOW = False
else:
    print('MLflow not available — running without tracking')

In [ ]:
# Metric helper functions
def wape(y_true, y_pred):
    """Weighted Absolute Percentage Error — standard for demand forecasting."""
    return np.sum(np.abs(y_true - y_pred)) / max(np.sum(np.abs(y_true)), 1)

def mase(y_true, y_pred, y_train, seasonality=12):
    """Mean Absolute Scaled Error (Hyndman & Koehler, 2006).
    Referenced in Petropoulos Section 2.12.2 as the recommended scale-free metric."""
    n = len(y_train)
    naive_errors = np.abs(y_train[seasonality:] - y_train[:-seasonality])
    scale = np.mean(naive_errors) if len(naive_errors) > 0 else 1.0
    if scale == 0:
        scale = 1.0
    return np.mean(np.abs(y_true - y_pred)) / scale

def bias_metric(y_true, y_pred):
    """Signed bias — positive means over-forecasting."""
    return np.mean(y_pred - y_true)

def compute_all_metrics(y_true, y_pred, y_train=None):
    metrics = {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'WAPE': wape(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
        'Bias': bias_metric(y_true, y_pred),
    }
    if y_train is not None and len(y_train) > 12:
        metrics['MASE'] = mase(y_true, y_pred, y_train)
    return metrics

print('Metric functions defined: RMSE, MAE, WAPE, R2, Bias, MASE')

## 2. Time-Series Split for Training

In [ ]:
# Temporal split
sorted_months = sorted(fg['month'].unique())
n_months = len(sorted_months)
train_end = sorted_months[min(23, n_months-7)]
val_end = sorted_months[min(29, n_months-1)]

train_df = fg[fg['month'] <= train_end].copy()
val_df   = fg[(fg['month'] > train_end) & (fg['month'] <= val_end)].copy()
test_df  = fg[fg['month'] > val_end].copy()

# Features for ML models
feature_cols = [
    'month_num', 'quarter', 'year', 'is_year_end', 'is_sl_peak',
    'month_sin', 'month_cos',
    'demand_lag_1', 'demand_lag_2', 'demand_lag_3', 'demand_lag_6', 'demand_lag_12',
    'demand_rmean_3', 'demand_rmean_6', 'demand_rstd_3', 'demand_rstd_6',
    'demand_rmin_3', 'demand_rmax_3', 'demand_rmin_6', 'demand_rmax_6',
    'demand_cv_6', 'demand_momentum',
    'fg_category_enc', 'fg_code_enc',
    'complement_co_promo_lag1', 'substitute_pressure_lag1',
]
for col in ['on_hand_inventory', 'lead_time_days', 'supplier_otif',
            'promotion_flag', 'holiday_flag', 'price_per_unit']:
    if col in fg.columns:
        feature_cols.append(col)

available_feats = [c for c in feature_cols if c in fg.columns]
TARGET = 'demand_units'

X_train = train_df[available_feats].values
y_train = train_df[TARGET].values
X_val = val_df[available_feats].values
y_val = val_df[TARGET].values

if len(test_df) > 0:
    X_test = test_df[available_feats].values
    y_test = test_df[TARGET].values
else:
    X_test, y_test = X_val, y_val  # fallback

assert len(train_df) <= MAX_TRAIN_ROWS, f'Train set too large ({len(train_df)} rows) — run NB02 monthly aggregation'

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Features: {len(available_feats)}')



## 3. Statistical Models

### 3.1 Seasonal Naive Baseline

> *"Always start with a naive or seasonal naive benchmark. Any model that cannot outperform it is not worth deploying."*  
> — Petropoulos et al. (2022), Section 2.3.1

In [ ]:
# 3.1 Seasonal Naive (same month last year)
snaive_preds = []
snaive_actuals = []

for sku in val_df['fg_code'].unique():
    sku_train = train_df[train_df['fg_code'] == sku].sort_values('month')
    sku_val = val_df[val_df['fg_code'] == sku].sort_values('month')
    
    for _, row in sku_val.iterrows():
        same_month_ly = sku_train[sku_train['month'].dt.month == row['month'].month]
        if len(same_month_ly) > 0:
            pred = same_month_ly['demand_units'].iloc[-1]
        else:
            pred = sku_train['demand_units'].mean()
        snaive_preds.append(pred)
        snaive_actuals.append(row['demand_units'])

snaive_metrics = compute_all_metrics(np.array(snaive_actuals), np.array(snaive_preds), y_train)

print('=== Seasonal Naive Baseline ===')
for k, v in snaive_metrics.items():
    print(f'  {k}: {v:.4f}')

results_log = [{'model': 'Seasonal Naive', **snaive_metrics, 'type': 'Baseline'}]

In [ ]:
# 3.2 ETS (Holt-Winters) — Petropoulos Section 2.3
if HAS_SM:
    ets_preds = []
    ets_actuals = []
    n_sku_ets = 0
    
    # Fit per-SKU ETS (sample 20 SKUs for speed)
    sample_skus = train_df.groupby('fg_code')['demand_units'].mean().nlargest(20).index
    
    for sku in sample_skus:
        try:
            sku_df = train_df.loc[train_df['fg_code'] == sku, ['month', 'demand_units']].copy()
            sku_df['month'] = sku_df['month'].dt.to_period('M').dt.to_timestamp()
            sku_train = sku_df.groupby('month')['demand_units'].sum().sort_index().asfreq('MS')
            sku_val = val_df[val_df['fg_code'] == sku].sort_values('month')
            
            if len(sku_train) < 24 or len(sku_val) == 0:
                continue
            
            model = ExponentialSmoothing(
                sku_train, seasonal_periods=12,
                trend='add', seasonal='add',
                damped_trend=True
            ).fit(optimized=True)
            
            forecast = model.forecast(len(sku_val))
            ets_preds.extend(forecast.values)
            ets_actuals.extend(sku_val['demand_units'].values)
            n_sku_ets += 1
        except Exception:
            continue
    
    if len(ets_preds) > 0:
        ets_metrics = compute_all_metrics(np.array(ets_actuals), np.array(ets_preds), y_train)
        print(f'=== ETS (Holt-Winters Additive Damped) — {n_sku_ets} SKUs ===')
        for k, v in ets_metrics.items():
            print(f'  {k}: {v:.4f}')
        results_log.append({'model': 'ETS (HW Damped)', **ets_metrics, 'type': 'Statistical'})
else:
    print('statsmodels not available — skipping ETS')

In [ ]:
# 3.3 Auto-ARIMA per SKU (pmdarima) with SARIMAX fallback
if HAS_SM:
    arima_preds, arima_actuals, arima_skus = [], [], []
    n_sku_arima = min(30, val_df['fg_code'].nunique())
    sample_skus = val_df.groupby('fg_code')['demand_units'].mean().nlargest(n_sku_arima).index

    for sku in sample_skus:
        sku_train = train_df[train_df['fg_code'] == sku].sort_values('month')['demand_units'].values
        sku_val = val_df[val_df['fg_code'] == sku].sort_values('month')
        if len(sku_train) < 14 or len(sku_val) == 0:
            continue
        horizon = len(sku_val)
        fc = auto_arima_forecast(sku_train, horizon)
        if fc is None:
            fc = sarimax_fallback_forecast(sku_train, horizon)
        if fc is None:
            fc = np.full(horizon, sku_train[-1])
        fc = cap_forecast(sku_train, fc)
        arima_preds.extend(fc[:horizon])
        arima_actuals.extend(sku_val['demand_units'].values)
        arima_skus.extend([sku] * horizon)

    if arima_preds:
        arima_metrics = evaluate_forecast_suite(
            np.array(arima_actuals), np.array(arima_preds), sku_ids=np.array(arima_skus))
        print(f'=== Auto-ARIMA + SARIMAX fallback — {len(sample_skus)} SKUs ===')
        for k, v in arima_metrics.items():
            print(f'  {k}: {v:.4f}')
        results_log.append({'model': 'Auto-ARIMA', **arima_metrics, 'type': 'Statistical'})
    else:
        print('Auto-ARIMA: no SKUs produced forecasts')
else:
    print('statsmodels/pmdarima not available — skipping Auto-ARIMA')



## 4. ML Models — Gradient Boosted Trees

> *"Gradient boosted trees have become the de-facto standard for tabular forecasting tasks [...] The M5 winners used LightGBM and achieved substantial improvements over statistical benchmarks."*  
> — Petropoulos et al. (2022), Section 2.7.4

### Why Tree-Based Models for Warehousing?
- Handle non-linear relationships (promotions, holidays)
- Native categorical support (CatBoost)
- Cross-learn across hundreds of SKUs simultaneously
- Quantile regression for uncertainty intervals (p10/p50/p90)

### 4.0 Hyperparameter Tuning (Optuna)

Timeboxed search on validation WAPE before final LightGBM training.


In [ ]:
# 4.0 Optuna tuning for LightGBM (validation WAPE)
OPTUNA_TRIALS = 25
USE_LOG_TARGET = True

y_train_fit = np.log1p(y_train) if USE_LOG_TARGET else y_train
y_val_fit = np.log1p(y_val) if USE_LOG_TARGET else y_val

try:
    import optuna
    print(f'Running Optuna ({OPTUNA_TRIALS} trials)...')
    lgb_params = tune_lightgbm_optuna(X_train, y_train_fit, X_val, y_val_fit, n_trials=OPTUNA_TRIALS, seed=SEED, n_jobs=LGB_THREADS)
    print('Best params:', lgb_params)
except Exception as e:
    print(f'Optuna unavailable ({e}) — using default LightGBM params')
    lgb_params = {
        'objective': 'regression', 'metric': 'rmse', 'learning_rate': 0.05,
        'num_leaves': 31, 'max_depth': 8, 'min_child_samples': 20,
        'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
        'reg_alpha': 0.1, 'reg_lambda': 0.1, 'n_estimators': LGB_N_ESTIMATORS,
        'verbose': -1, 'n_jobs': LGB_THREADS, 'random_state': SEED,
    }



In [ ]:
# 4.1 LightGBM — tuned params + log-target training
# lgb_params set by Optuna cell above; USE_LOG_TARGET defined there
train_weights = sku_sample_weights(train_df, 'fg_code', TARGET)

t0 = time.time()
lgb_model = lgb.LGBMRegressor(**lgb_params)
lgb_model.fit(
    X_train, y_train_fit,
    sample_weight=train_weights,
    eval_set=[(X_val, y_val_fit)],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
lgb_time = time.time() - t0

lgb_pred_val = lgb_model.predict(X_val)
if USE_LOG_TARGET:
    lgb_pred_val = np.expm1(lgb_pred_val)
lgb_pred_val = np.clip(lgb_pred_val, 0, None)
lgb_metrics = compute_all_metrics(y_val, lgb_pred_val, y_train)

print(f'=== LightGBM tuned + log-target (Validation) — {lgb_time:.1f}s ===')
for k, v in lgb_metrics.items():
    print(f'  {k}: {v:.4f}')
results_log.append({'model': 'LightGBM-tuned', **lgb_metrics, 'type': 'ML', 'train_time': lgb_time})

if HAS_MLFLOW:
    with mlflow.start_run(run_name='LightGBM-tuned-v6'):
        mlflow.log_params({k: v for k, v in lgb_params.items() if isinstance(v, (int, float, str, bool))})
        mlflow.log_param('use_log_target', USE_LOG_TARGET)
        mlflow.log_metrics({k: float(v) for k, v in lgb_metrics.items() if isinstance(v, (int, float, np.floating))})
        mlflow_log_model_safe(lgb_model, 'model', flavor='lightgbm')



In [ ]:
# 4.1b Ablation: uniform vs inverse-demand sample weights (log-target)
m_uni = lgb.LGBMRegressor(**{**lgb_params, 'n_estimators': min(200, lgb_params.get('n_estimators', 300))})
m_uni.fit(X_train, y_train_fit, eval_set=[(X_val, y_val_fit)], callbacks=[lgb.early_stopping(30, verbose=False)])
p_uni = np.expm1(m_uni.predict(X_val)) if USE_LOG_TARGET else m_uni.predict(X_val)
p_uni = np.clip(p_uni, 0, None)
wape_uni = wape(y_val, p_uni)
wape_wt = wape(y_val, lgb_pred_val)
print(f'Sample-weight ablation — uniform: WAPE={wape_uni:.4f}, inverse-demand: WAPE={wape_wt:.4f}')



In [ ]:
# 4.2 CatBoost
cat_params = {
    'iterations': CAT_ITERATIONS,
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 3,
    'random_seed': SEED,
    'verbose': 0, 'thread_count': LGB_THREADS,
    'early_stopping_rounds': 50
}

t0 = time.time()
cat_model = CatBoostRegressor(**cat_params)
cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=0)
cat_time = time.time() - t0

cat_pred_val = np.clip(cat_model.predict(X_val), 0, None)
cat_metrics = compute_all_metrics(y_val, cat_pred_val, y_train)

print(f'=== CatBoost (Validation) — trained in {cat_time:.1f}s ===')
for k, v in cat_metrics.items():
    print(f'  {k}: {v:.4f}')

results_log.append({'model': 'CatBoost', **cat_metrics, 'type': 'ML', 'train_time': cat_time})

if HAS_MLFLOW:
    with mlflow.start_run(run_name='CatBoost-v6'):
        mlflow.log_params({k: str(v) for k, v in cat_params.items()})
        mlflow.log_metrics({k: v for k, v in cat_metrics.items()})
        mlflow.log_metric('train_time_sec', cat_time)
        mlflow_log_model_safe(cat_model, 'model', flavor='catboost')
        print('  Logged to MLflow')


In [ ]:
# 4.3 XGBoost
xgb_params = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.05,
    'max_depth': 8,
    'n_estimators': XGB_N_ESTIMATORS,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': SEED,
    'verbosity': 0, 'n_jobs': LGB_THREADS,
    'early_stopping_rounds': 50
}

t0 = time.time()
xgb_model = xgb.XGBRegressor(**xgb_params)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_time = time.time() - t0

xgb_pred_val = np.clip(xgb_model.predict(X_val), 0, None)
xgb_metrics = compute_all_metrics(y_val, xgb_pred_val, y_train)

print(f'=== XGBoost (Validation) — trained in {xgb_time:.1f}s ===')
for k, v in xgb_metrics.items():
    print(f'  {k}: {v:.4f}')

results_log.append({'model': 'XGBoost', **xgb_metrics, 'type': 'ML', 'train_time': xgb_time})

if HAS_MLFLOW:
    with mlflow.start_run(run_name='XGBoost-v6'):
        mlflow.log_params({k: str(v) for k, v in xgb_params.items()})
        mlflow.log_metrics({k: v for k, v in xgb_metrics.items()})
        mlflow.log_metric('train_time_sec', xgb_time)
        mlflow_log_model_safe(xgb_model, 'model', flavor='xgboost')
        print('  Logged to MLflow')


## 5. Quantile Regression (Uncertainty Estimation)

Enterprise WMS requires **prediction intervals** (p10, p50, p90) for:
- **p50** — median forecast for slotting velocity
- **p90** — safety stock calculation (serve 90% of demand scenarios)
- **p90 - p10 spread** — demand volatility for GA placement

> *"Probabilistic forecasts are far more useful than point forecasts for inventory management."*  
> — Petropoulos et al. (2022), Section 2.11

In [ ]:
# 5.1 LightGBM Quantile Regression
quantile_models = {}

for q, alpha in [('p10', 0.10), ('p50', 0.50), ('p90', 0.90)]:
    qr_params = {
        'objective': 'quantile',
        'alpha': alpha,
        'metric': 'quantile',
        'learning_rate': 0.05,
        'num_leaves': 63,
        'max_depth': 8,
        'n_estimators': 200, 'n_jobs': LGB_THREADS,
        'verbose': -1,
        'random_state': SEED
    }
    
    model = lgb.LGBMRegressor(**qr_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(30, verbose=False)])
    quantile_models[q] = model

# Generate quantile forecasts
q_preds = {}
for q, model in quantile_models.items():
    q_preds[q] = np.clip(model.predict(X_val), 0, None)

# Visualise quantile forecasts for one SKU
sku_idx = val_df['fg_code'] == val_df['fg_code'].unique()[0]
sku_months = val_df.loc[sku_idx, 'month']
sku_actual = val_df.loc[sku_idx, 'demand_units']

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(sku_months, sku_actual, 'ko-', label='Actual', markersize=6)
ax.plot(sku_months, q_preds['p50'][sku_idx], 'b-', label='p50 (median)', linewidth=2)
ax.fill_between(sku_months,
                q_preds['p10'][sku_idx],
                q_preds['p90'][sku_idx],
                alpha=0.3, color='blue', label='p10-p90 interval')
ax.set_title(f'Quantile Forecast — {val_df["fg_code"].unique()[0]}')
ax.set_ylabel('Demand (units)')
ax.legend()
plt.tight_layout()
plt.show()

# Coverage check
in_interval = ((y_val >= q_preds['p10']) & (y_val <= q_preds['p90'])).mean()
print(f'\n80% Prediction Interval Coverage: {in_interval:.1%} (target: 80%)')


## 6. Time-Series Cross-Validation

> *"Expanding window CV avoids look-ahead bias and provides robust performance estimates."*  
> — Petropoulos et al. (2022), Section 2.7.5

In [ ]:
# 6.1 Time-Series CV with expanding window
tscv = TimeSeriesSplit(n_splits=2)  # 2 folds — lower RAM
X_all = fg[available_feats].values
y_all = fg[TARGET].values

cv_results = {'LightGBM': [], 'XGBoost': [], 'CatBoost': []}

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_all)):
    X_tr, X_va = X_all[train_idx], X_all[val_idx]
    y_tr, y_va = y_all[train_idx], y_all[val_idx]
    
    # LightGBM
    m = lgb.LGBMRegressor(**{**lgb_params, 'n_estimators': 150, 'n_jobs': LGB_THREADS, 'verbose': -1})
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(30, verbose=False)])
    pred = np.clip(m.predict(X_va), 0, None)
    cv_results['LightGBM'].append(wape(y_va, pred))
    
    # XGBoost
    m = xgb.XGBRegressor(**{**xgb_params, 'n_estimators': 150, 'n_jobs': LGB_THREADS})
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    pred = np.clip(m.predict(X_va), 0, None)
    cv_results['XGBoost'].append(wape(y_va, pred))
    
    # CatBoost
    m = CatBoostRegressor(**{**cat_params, 'iterations': 150, 'thread_count': LGB_THREADS})
    m.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
    pred = np.clip(m.predict(X_va), 0, None)
    cv_results['CatBoost'].append(wape(y_va, pred))
    
    print(f'Fold {fold+1}: LGB={cv_results["LightGBM"][-1]:.4f}, XGB={cv_results["XGBoost"][-1]:.4f}, CAT={cv_results["CatBoost"][-1]:.4f}')

print('\n=== Time-Series CV Summary (WAPE) ===')
for name, scores in cv_results.items():
    print(f'  {name}: mean={np.mean(scores):.4f} +/- {np.std(scores):.4f}')


## 6b. Rolling-Origin Evaluation

Multiple train/val cut points reduce brittleness of a single static split.


In [ ]:
# 6b. Rolling-origin WAPE (LightGBM tuned)
origins = rolling_origin_splits(sorted_months, n_origins=4, val_horizon=6, min_train_months=18)
rolling_wapes = []
for spec in origins:
    tr = fg[fg['month'] <= spec['train_end']]
    va = fg[(fg['month'] > spec['train_end']) & (fg['month'] <= spec['val_end'])]
    if len(va) == 0:
        continue
    Xtr, ytr = tr[available_feats].values, np.log1p(tr[TARGET].values)
    Xva, yva = va[available_feats].values, va[TARGET].values
    w = sku_sample_weights(tr, 'fg_code', TARGET)
    m = lgb.LGBMRegressor(**{**lgb_params, 'n_estimators': 150, 'verbose': -1, 'n_jobs': LGB_THREADS})
    m.fit(Xtr, ytr, sample_weight=w)
    pred = np.clip(np.expm1(m.predict(Xva)), 0, None)
    rolling_wapes.append(wape(yva, pred))
    print(f"  Origin train_end={spec['train_end'].strftime('%Y-%m')}: WAPE={rolling_wapes[-1]:.4f}")

if rolling_wapes:
    print(f'Rolling-origin WAPE: mean={np.mean(rolling_wapes):.4f}, std={np.std(rolling_wapes):.4f}')
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.boxplot(rolling_wapes, labels=['LightGBM'])
    ax.set_title('Rolling-Origin WAPE Distribution')
    ax.set_ylabel('WAPE')
    plt.tight_layout()
    plt.show()



## 7. Feature Importance

In [ ]:
# 7.1 LightGBM Feature Importance (gain-based)
importance = pd.DataFrame({
    'feature': available_feats,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, max(6, len(importance)*0.3)))
ax.barh(importance['feature'][:20], importance['importance'][:20])
ax.set_xlabel('Feature Importance (Split Count)')
ax.set_title('LightGBM — Top 20 Features')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\nTop 5 features:')
for _, row in importance.head(5).iterrows():
    print(f'  {row["feature"]}: {row["importance"]}')

## 8. Model Comparison Summary

In [ ]:
# 8.1 Results comparison table
results_df = pd.DataFrame(results_log)
display_cols = ['model', 'type', 'RMSE', 'MAE', 'WAPE', 'WAPE_median_per_sku', 'R2', 'Bias']
display_cols = [c for c in display_cols if c in results_df.columns]

print('=== Model Comparison (Validation Set) ===')
print(results_df[display_cols].to_string(index=False, float_format='%.4f'))

# Visual comparison — use robust WAPE for chart (median per-SKU when available)
plot_df = results_df.copy()
if 'WAPE_median_per_sku' in plot_df.columns:
    plot_df['WAPE_plot'] = plot_df['WAPE_median_per_sku'].fillna(plot_df['WAPE'])
else:
    plot_df['WAPE_plot'] = plot_df['WAPE']
# Cap display metrics so one blown statistical forecast cannot hide ML models
wape_cap = np.nanpercentile(plot_df['WAPE_plot'].replace([np.inf, -np.inf], np.nan).dropna(), 75) * 1.5
wape_cap = max(wape_cap, 0.5)
plot_df['WAPE_capped'] = plot_df['WAPE_plot'].clip(upper=wape_cap)
rmse_cap = np.nanpercentile(plot_df['RMSE'].replace([np.inf, -np.inf], np.nan).dropna(), 75) * 1.5
plot_df['RMSE_capped'] = plot_df['RMSE'].clip(upper=max(rmse_cap, plot_df['RMSE'].median() * 3))
r2_floor = np.nanpercentile(plot_df['R2'].replace([np.inf, -np.inf], np.nan).dropna(), 25) - 0.5
plot_df['R2_capped'] = plot_df['R2'].clip(lower=r2_floor)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#e74c3c' if t == 'Baseline' else '#3498db' if t == 'ML' else '#27ae60' for t in plot_df['type']]
axes[0].barh(plot_df['model'], plot_df['WAPE_capped'], color=colors)
axes[0].set_title('WAPE (median per-SKU, capped for viz)')
axes[0].axvline(0.10, color='green', linestyle='--', alpha=0.5, label='Target (10%)')
axes[0].legend()
axes[1].barh(plot_df['model'], plot_df['RMSE_capped'], color=colors)
axes[1].set_title('RMSE (capped for viz)')
axes[2].barh(plot_df['model'], plot_df['R2_capped'], color=colors)
axes[2].set_title('R2 (floored for viz)')
plt.suptitle('Model Performance Comparison (robust scaling)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Full table for statistical models that were capped in chart
if (plot_df['WAPE_plot'] > wape_cap).any():
    print('\nNote: Auto-ARIMA pooled WAPE can explode on short monthly series (SARIMAX blow-up).')
    print('Use WAPE_median_per_sku for fair statistical vs ML comparison.')
    print(plot_df.loc[plot_df['WAPE_plot'] > wape_cap, ['model', 'WAPE', 'WAPE_median_per_sku']].to_string(index=False))

# Best model (robust metric)
rank_col = 'WAPE_median_per_sku' if 'WAPE_median_per_sku' in results_df.columns else 'WAPE'
best = results_df.loc[results_df[rank_col].idxmin()]
print(f'\nBest model ({rank_col}): {best["model"]} ({rank_col}: {best[rank_col]:.4f})')

# Save results
results_df.to_csv(ENG_DIR / 'model_comparison_results.csv', index=False)
print(f'\nNext: Notebook 04 — Detailed Model Evaluation & Diagnostics')